In [ ]:
import pandas as pd
import numpy as np
import os
import glob

INPUT_DIR = os.path.join('..', 'Inputs')
print('Input directory:', os.path.abspath(INPUT_DIR))
print('Files:', sorted(os.listdir(INPUT_DIR)))

In [ ]:
# ── Load GIS Generators (EIA 860 data) ──────────────────────────────────────
gis_gen = pd.read_csv(os.path.join(INPUT_DIR, 'GIS_generators_2026_January.csv'), encoding='latin1')
print(f'GIS Generators: {len(gis_gen)} rows, {len(gis_gen.columns)} cols')
print(f'  Columns: {list(gis_gen.columns)}')

# ── Load Plant coordinates ───────────────────────────────────────────────────
gis_plant = pd.read_csv(os.path.join(INPUT_DIR, 'GIS_Plant_madeup_updated.csv'), encoding='latin1')
print(f'\nGIS Plants: {len(gis_plant)} rows, {len(gis_plant.columns)} cols')
print(f'  Columns: {list(gis_plant.columns)}')

# ── Load ERCOT GIS Report (Large Gen + Small Gen sheets) ─────────────────────
xlsx_files = glob.glob(os.path.join(INPUT_DIR, 'RPT*.xlsx'))
if not xlsx_files:
    raise FileNotFoundError('No RPT*.xlsx file found in Inputs/')
xlsx_path = xlsx_files[0]
print(f'\nERCOT GIS Report: {os.path.basename(xlsx_path)}')

# Large Gen: header at row 30 (0-indexed)
ercot_large = pd.read_excel(xlsx_path, sheet_name='Project Details - Large Gen', header=30)
print(f'  Large Gen: {len(ercot_large)} rows, {len(ercot_large.columns)} cols')

# Small Gen: header at row 14 (0-indexed)
ercot_small = pd.read_excel(xlsx_path, sheet_name='Project Details - Small Gen', header=14)
print(f'  Small Gen: {len(ercot_small)} rows, {len(ercot_small.columns)} cols')

# Combine Large + Small into one ERCOT GIS DataFrame
ercot_gis = pd.concat([ercot_large, ercot_small], ignore_index=True)
print(f'  Combined ERCOT GIS: {len(ercot_gis)} rows')
print(f'  Columns: {list(ercot_gis.columns)}')

# ── Load fuel type key ───────────────────────────────────────────────────────
key_gen = pd.read_csv(os.path.join(INPUT_DIR, 'key_gen_type.csv'), encoding='latin1')
print(f'\nKey Gen Type: {len(key_gen)} rows')
print(key_gen.to_string(index=False))

In [ ]:
# ── Step 1: Merge generators + plant coordinates on Utility ID ────────────────
# gis_gen has generator-level rows; gis_plant has plant-level lat/lon
df = gis_gen.merge(
    gis_plant[['Utility ID', 'Plant Code', 'Latitude', 'Longitude']],
    on='Utility ID',
    how='left',
    suffixes=('', '_plant')
)
print(f'After plant merge: {len(df)} rows')
print(f'  With coordinates: {df["Latitude"].notna().sum()}')
print(f'  Missing coordinates: {df["Latitude"].isna().sum()}')

# ── Step 2: Map fuel types using key_gen_type ────────────────────────────────
fuel_map = key_gen.set_index(['Prime Mover', 'Energy Source 1'])['Technology']
df['Technology_Mapped'] = df.set_index(['Prime Mover', 'Energy Source 1']).index.map(
    lambda x: fuel_map.get(x, 'Other')
)
print(f'\nTechnology mapping:')
print(df['Technology_Mapped'].value_counts().to_string())

# ── Step 3: Merge with ERCOT GIS Report on Utility ID = INR ──────────────────
# Select useful columns from ERCOT GIS report
ercot_keep = ['INR', 'Project Name', 'CDR Reporting Zone', 'POI Location',
              'Fuel', 'Technology', 'Capacity (MW)', 'IA Signed',
              'Approved for Energization', 'Approved for Synchronization',
              'GIM Study Phase']
# Only keep columns that actually exist
ercot_keep = [c for c in ercot_keep if c in ercot_gis.columns]
print(f'\nKeeping ERCOT columns: {ercot_keep}')

ercot_subset = ercot_gis[ercot_keep].copy()
# Strip whitespace from INR and Utility ID for clean join
ercot_subset['INR'] = ercot_subset['INR'].astype(str).str.strip()
df['Utility ID'] = df['Utility ID'].astype(str).str.strip()

# Deduplicate ERCOT data — keep first entry per INR to avoid row explosion
ercot_subset = ercot_subset.drop_duplicates(subset='INR', keep='first')
# Drop rows where INR is null/nan
ercot_subset = ercot_subset[ercot_subset['INR'].notna() & (ercot_subset['INR'] != 'nan')]
print(f'ERCOT unique INRs: {len(ercot_subset)}')

# Rename INR → Utility ID for merge
ercot_subset = ercot_subset.rename(columns={'INR': 'Utility ID'})

df = df.merge(ercot_subset, on='Utility ID', how='left', suffixes=('', '_ercot'))
print(f'\nAfter ERCOT GIS merge: {len(df)} rows')
print(f'  With ERCOT data: {df["CDR Reporting Zone"].notna().sum() if "CDR Reporting Zone" in df.columns else "N/A"}')

# ── Step 4: Filter to approved generators (IA Signed is not null) ────────────
print(f'\n── Approval Filter ──')
print(f'  Total generators before filter: {len(df)}')
print(f'  With IA Signed: {df["IA Signed"].notna().sum()}')
df_approved = df[df['IA Signed'].notna()].copy()
print(f'  Approved generators (IA Signed): {len(df_approved)}')

print(f'\nFinal combined DataFrame: {len(df_approved)} rows, {len(df_approved.columns)} cols')
print(f'Columns: {list(df_approved.columns)}')
df_approved.head(10)

In [ ]:
df

In [ ]:
# ── Summary Statistics (Approved Generators Only) ─────────────────────────────
print('=' * 70)
print('COMBINED GIS GENERATOR DATASET SUMMARY  (IA Signed only)')
print('=' * 70)

# Ensure MW column is numeric
df_approved['Nameplate Capacity (MW)'] = pd.to_numeric(df_approved['Nameplate Capacity (MW)'], errors='coerce')

print(f'\nTotal approved generators: {len(df_approved)}')
if 'Nameplate Capacity (MW)' in df_approved.columns:
    print(f'Total nameplate capacity: {df_approved["Nameplate Capacity (MW)"].sum():,.0f} MW')

print(f'\n── By Technology ──')
tech_summary = df_approved.groupby('Technology_Mapped').agg(
    Count=('Technology_Mapped', 'size'),
    Total_MW=('Nameplate Capacity (MW)', 'sum'),
    Avg_MW=('Nameplate Capacity (MW)', 'mean')
).sort_values('Total_MW', ascending=False)
print(tech_summary.to_string())

print(f'\n── By CDR Reporting Zone ──')
if 'CDR Reporting Zone' in df_approved.columns:
    zone_summary = df_approved.groupby('CDR Reporting Zone').agg(
        Count=('CDR Reporting Zone', 'size'),
        Total_MW=('Nameplate Capacity (MW)', 'sum')
    ).sort_values('Total_MW', ascending=False)
    print(zone_summary.to_string())

print(f'\n── By County (top 20) ──')
county_summary = df_approved.groupby('County').agg(
    Count=('County', 'size'),
    Total_MW=('Nameplate Capacity (MW)', 'sum')
).sort_values('Total_MW', ascending=False).head(20)
print(county_summary.to_string())

print(f'\n── Coordinate Coverage ──')
print(f'  With lat/lon: {df_approved["Latitude"].notna().sum()} ({100*df_approved["Latitude"].notna().mean():.1f}%)')
print(f'  Missing lat/lon: {df_approved["Latitude"].isna().sum()} ({100*df_approved["Latitude"].isna().mean():.1f}%)')

print(f'\n── Data Quality ──')
for col in ['Nameplate Capacity (MW)', 'Energy Source 1', 'Prime Mover', 'County', 'Latitude', 'CDR Reporting Zone']:
    if col in df_approved.columns:
        n_missing = df_approved[col].isna().sum()
        print(f'  {col}: {n_missing} missing ({100*n_missing/len(df_approved):.1f}%)')

print('\n' + '=' * 70)